# `DimRedValidator` — оценка стабильности кластеров при снижении размерности

Класс для проверки того, насколько устойчива структура, которую мы видим на 2D-эмбеддингах
(UMAP / t-SNE / PaCMAP / LocalMAP). Строит несколько эмбеддингов с разными `random_state`
и позволяет визуально и количественно сравнить их.

Интерфейс гибкий: на вход передаётся класс эмбеддера и класс кластеризатора с их параметрами,
так что метод DR легко менять. Поддерживает:

- `fit` — обучает несколько эмбеддингов (либо принимает готовые — удобно для сравнения разных методов);
- `draw_embeddings` / `draw_clusterized` / `draw_colored` — интерактивные панели Plotly
  (раскраска по кластерам первого эмбеддинга / по координатам);
- `compute_stability` — матрица попарных ARI между кластеризациями + сводная статистика;
- `draw_least_similar`, `draw_diagnostic` (раскраска по признакам), `draw_subclusters`.

Главная цель — не принять случайный артефакт конкретного запуска за реальную структуру данных.

In [ ]:
# !pip install umap-learn hdbscan plotly -q

In [2]:
import time
from typing import Optional, List, Dict, Any, Callable

import numpy as np
import pandas as pd

from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
class DimRedValidator:
    """
    Validator for assessing cluster stability across multiple DR embeddings.

    Creates several embeddings using different random states and allows
    visual comparison to assess how stable the discovered structure is.

    Parameters
    ----------
    embedder_class : Callable
        Dimensionality reduction class (e.g., UMAP, TSNE).
        Must have fit_transform(X) method and accept random_state parameter.

    embedder_kws : dict, optional
        Keyword arguments for the embedder (e.g., {'n_neighbors': 15}).

    clusterer_class : Callable, optional
        Clustering class (e.g., HDBSCAN, KMeans).
        Must have fit_predict(X) method.

    clusterer_kws : dict, optional
        Keyword arguments for the clusterer.

    random_states : list[int], optional
        List of random seeds for creating multiple embeddings.
        Default: [0, 1, 2, 3, 4]

    Attributes
    ----------
    embeddings : list[np.ndarray]
        List of 2D embeddings, each of shape (n_samples, 2).

    clusterings : list[np.ndarray]
        List of cluster labels, each of shape (n_samples,).

    Example
    -------
    >>> validator = DimRedValidator(
    ...     embedder_class=UMAP,
    ...     embedder_kws={'n_neighbors': 15},
    ...     clusterer_class=HDBSCAN,
    ...     clusterer_kws={'min_cluster_size': 30},
    ...     random_states=[0, 1, 2]
    ... )
    >>> validator.fit(X)
    >>> validator.draw_embeddings([0, 1, 2])
    """

    def __init__(
        self,
        embedder_class: Callable,
        embedder_kws: Optional[Dict[str, Any]] = None,
        clusterer_class: Optional[Callable] = None,
        clusterer_kws: Optional[Dict[str, Any]] = None,
        random_states: Optional[List[int]] = None,
    ):
        self.embedder_class = embedder_class
        self.embedder_kws = embedder_kws or {}
        self.clusterer_class = clusterer_class
        self.clusterer_kws = clusterer_kws or {}
        self.random_states = random_states if random_states is not None else [0, 1, 2, 3, 4]

        # Will be populated after fit()
        self.embeddings: List[np.ndarray] = []
        self.clusterings: List[np.ndarray] = []
        self.data: Optional[np.ndarray] = None

    def fit(
        self,
        data: np.ndarray,
        precomputed_embeddings: Optional[List[np.ndarray]] = None
    ) -> 'DimRedValidator':
        """
        Compute embeddings and clusterings.

        Parameters
        ----------
        data : np.ndarray of shape (n_samples, n_features)
            Input high-dimensional data.

        precomputed_embeddings : list[np.ndarray], optional
            Pre-computed embeddings to use instead of computing new ones.
            Useful for comparing different DR methods.

        Returns
        -------
        self : DimRedValidator
        """
        self.data = np.array(data)

        # Either use precomputed or compute new embeddings
        if precomputed_embeddings is not None:
            self.embeddings = [np.array(e) for e in precomputed_embeddings]
            print(f"Loaded {len(self.embeddings)} precomputed embeddings")
        else:
            self.embeddings = []
            for i, rs in enumerate(self.random_states):
                print(f"Computing embedding {i+1}/{len(self.random_states)} (rs={rs})...", end=" ")
                embedder = self.embedder_class(random_state=rs, **self.embedder_kws)
                emb = embedder.fit_transform(self.data)
                self.embeddings.append(emb)
                print("done")

        # Compute clusterings if clusterer is provided
        if self.clusterer_class is not None:
            self.clusterings = []
            for emb in self.embeddings:
                clusterer = self.clusterer_class(**self.clusterer_kws)
                labels = clusterer.fit_predict(emb)
                self.clusterings.append(labels)

            n_clusters = len(set(self.clusterings[0])) - (1 if -1 in self.clusterings[0] else 0)
            print(f"Reference clustering: {n_clusters} clusters found")

        return self

    def draw_embeddings(
        self,
        indices: Optional[List[int]] = None,
        hover_data: Optional[Dict[str, np.ndarray]] = None,
        point_size: int = 3,
        height: int = 400
    ) -> go.Figure:
        """
        Draw embeddings as scatter plots (no coloring).

        Parameters
        ----------
        indices : list[int], optional
            Which embeddings to plot. Default: first 3.

        hover_data : dict[str, np.ndarray], optional
            Additional data to show on hover.

        point_size : int
            Size of scatter points.

        height : int
            Height of each subplot in pixels.

        Returns
        -------
        go.Figure
            Plotly figure with subplots.
        """
        indices = indices if indices is not None else list(range(min(3, len(self.embeddings))))

        fig = make_subplots(
            rows=1, cols=len(indices),
            subplot_titles=[f"Embedding {i}" for i in indices]
        )

        for col, idx in enumerate(indices, 1):
            emb = self.embeddings[idx]

            if hover_data:
                hover_text = [
                    '<br>'.join([f"{k}: {v[i]}" for k, v in hover_data.items()])
                    for i in range(len(emb))
                ]
                hovertemplate = "x: %{x:.2f}<br>y: %{y:.2f}<br>%{text}<extra></extra>"
            else:
                hover_text = None
                hovertemplate = "x: %{x:.2f}<br>y: %{y:.2f}<extra></extra>"

            fig.add_trace(
            go.Scatter(
                x=emb[:, 0],
                y=emb[:, 1],
                mode='markers',
                marker=dict(size=point_size, opacity=0.6),
                text=hover_text,
                hovertemplate=hovertemplate
            ),
            row=1, col=col
        )

        fig.update_layout(
            height=height,
            width=350 * len(indices),
            showlegend=False
        )
        fig.show()
        return fig

    def draw_clusterized(
        self,
        indices: Optional[List[int]] = None,
        hover_data: Optional[Dict[str, np.ndarray]] = None,
        point_size: int = 3,
        height: int = 400
    ) -> go.Figure:
        """
        Draw embeddings colored by clusters from the FIRST embedding.

        This shows whether cluster structure is preserved across
        different random initializations.

        Parameters
        ----------
        indices : list[int], optional
            Which embeddings to plot. Default: first 3.

        hover_data : dict[str, np.ndarray], optional
            Additional data to show on hover.

        point_size : int
            Size of scatter points.

        height : int
            Height of each subplot in pixels.

        Returns
        -------
        go.Figure
        """
        if not self.clusterings:
            raise ValueError("No clustering computed. Provide clusterer_class in __init__.")

        indices = indices if indices is not None else list(range(min(3, len(self.embeddings))))
        ref_labels = self.clusterings[0]  # Reference: clusters from first embedding

        fig = make_subplots(
            rows=1, cols=len(indices),
            subplot_titles=[f"Embedding {i} (ref clusters)" for i in indices]
        )

        for col, idx in enumerate(indices, 1):
            emb = self.embeddings[idx]

            # Build hover text
            if hover_data:
                hover_text = [
                    '<br>'.join([f"{k}: {v[i]}" for k, v in hover_data.items()])
                    for i in range(len(emb))
                ]
                hovertemplate = "cluster: %{marker.color}<br>%{text}<extra></extra>"
            else:
                hover_text = None
                hovertemplate = "cluster: %{marker.color}<extra></extra>"


            fig.add_trace(
                go.Scatter(
                    x=emb[:, 0],
                    y=emb[:, 1],
                    mode='markers',
                    marker=dict(
                        size=point_size,
                        opacity=0.6,
                        color=ref_labels,
                        colorscale='Rainbow'
                    ),
                    text=hover_text,
                    hovertemplate=hovertemplate
                ),
                row=1, col=col
            )

        fig.update_layout(height=height, width=350 * len(indices), showlegend=False)
        fig.show()
        return fig

    def draw_colored(
        self,
        indices: Optional[List[int]] = None,
        hover_data: Optional[Dict[str, np.ndarray]] = None,
        point_size: int = 3,
        height: int = 400
    ) -> go.Figure:
        """
        Draw embeddings colored by coordinates of the FIRST embedding.

        Color mapping: (x, y) -> RGB where R=x, G=y, B=1-x (normalized).
        This reveals how points "move" between different embeddings.

        Parameters
        ----------
        indices : list[int], optional
            Which embeddings to plot. Default: first 3.

        hover_data : dict[str, np.ndarray], optional
            Additional data to show on hover.

        point_size : int
            Size of scatter points.

        height : int
            Height of each subplot in pixels.

        Returns
        -------
        go.Figure
        """
        indices = indices if indices is not None else list(range(min(3, len(self.embeddings))))
        ref_emb = self.embeddings[0]

        # Normalize reference coordinates to [0, 1]
        x_norm = (ref_emb[:, 0] - ref_emb[:, 0].min()) / (np.ptp(ref_emb[:, 0]) + 1e-10)
        y_norm = (ref_emb[:, 1] - ref_emb[:, 1].min()) / (np.ptp(ref_emb[:, 1]) + 1e-10)

        # Create RGB colors
        colors = [
            f'rgb({int(255*r)},{int(255*g)},{int(255*(1-r))})'
            for r, g in zip(x_norm, y_norm)
        ]

        fig = make_subplots(
            rows=1, cols=len(indices),
            subplot_titles=[f"Embedding {i} (ref coords)" for i in indices]
        )

        for col, idx in enumerate(indices, 1):
            emb = self.embeddings[idx]

            if hover_data:
                hover_text = [
                    '<br>'.join([f"{k}: {v[i]}" for k, v in hover_data.items()])
                    for i in range(len(emb))
                ]
                hovertemplate = "x: %{x:.2f}<br>y: %{y:.2f}<br>%{text}<extra></extra>"
            else:
                hover_text = None
                hovertemplate = "x: %{x:.2f}<br>y: %{y:.2f}<extra></extra>"

            fig.add_trace(
            go.Scatter(
                x=emb[:, 0],
                y=emb[:, 1],
                mode='markers',
                marker=dict(size=point_size, opacity=0.6, color=colors),
                text=hover_text,
                hovertemplate=hovertemplate
            ),
            row=1, col=col
        )

        fig.update_layout(height=height, width=350 * len(indices), showlegend=False)
        fig.show()
        return fig


    def compute_stability(self) -> Dict[str, Any]:
        """
        Compute pairwise ARI between all clusterings.

        Returns
        -------
        dict with keys:
            'ari_matrix': np.ndarray of shape (n_embeddings, n_embeddings)
            'mean_ari': float - mean of upper triangle
            'std_ari': float - std of upper triangle
            'min_ari': float
            'max_ari': float (excluding diagonal)
        """
        if not self.clusterings:
            raise ValueError("No clusterings. Provide clusterer_class.")

        n = len(self.clusterings)
        ari_matrix = np.zeros((n, n))

        for i in range(n):
            for j in range(n):
                ari_matrix[i, j] = adjusted_rand_score(
                    self.clusterings[i],
                    self.clusterings[j]
                )

        # Upper triangle without diagonal
        upper = ari_matrix[np.triu_indices(n, k=1)]

        return {
            'ari_matrix': ari_matrix,
            'mean_ari': float(np.mean(upper)),
            'std_ari': float(np.std(upper)),
            'min_ari': float(np.min(upper)),
            'max_ari': float(np.max(upper)),
        }

    def find_least_similar(self) -> tuple:
        """
        Find two embeddings with least similar clusterings.

        Compares all pairs of clusterings using Adjusted Rand Index
        and returns the pair with minimum ARI.

        Returns
        -------
        tuple
            (idx1, idx2, ari_score) where idx1, idx2 are embedding indices
            and ari_score is their ARI (lower = less similar).

        Raises
        ------
        ValueError
            If no clusterings have been computed.
        """
        stability = self.compute_stability()
        ari_matrix = stability['ari_matrix']

        # Set diagonal to inf to exclude self-comparison
        ari_copy = ari_matrix.copy()
        np.fill_diagonal(ari_copy, np.inf)

        # Find minimum
        min_idx = np.unravel_index(np.argmin(ari_copy), ari_copy.shape)
        min_ari = ari_matrix[min_idx]

        return int(min_idx[0]), int(min_idx[1]), float(min_ari)


    def draw_least_similar(
        self,
        hover_data: Optional[Dict[str, np.ndarray]] = None,
        point_size: int = 3,
        height: int = 400
    ) -> go.Figure:
        """
        Draw the two embeddings with least similar clusterings side by side.

        Useful for identifying which random states produce the most
        different cluster structures.

        Parameters
        ----------
        hover_data : dict[str, np.ndarray], optional
            Additional data to show on hover.
        point_size : int
            Size of scatter points.
        height : int
            Height of figure in pixels.

        Returns
        -------
        go.Figure
            Plotly figure with two subplots.

        Raises
        ------
        ValueError
            If no clusterings have been computed.
        """
        idx1, idx2, ari = self.find_least_similar()
        print(f"Least similar pair: embeddings {idx1} & {idx2}, ARI = {ari:.4f}")

        return self.draw_clusterized(
            indices=[idx1, idx2],
            hover_data=hover_data,
            point_size=point_size,
            height=height
        )


    def draw_diagnostic(
        self,
        embedding_idx: int = 0,
        feature_indices: Optional[List[int]] = None,
        feature_names: Optional[List[str]] = None,
        point_size: int = 3,
        height: int = 350
    ) -> go.Figure:
        """
        Draw embedding colored by original feature values.

        Shows how different input features map onto the embedding space.
        Useful for understanding what drives the structure.

        Parameters
        ----------
        embedding_idx : int
            Which embedding to visualize.
        feature_indices : list[int], optional
            Which features from original data to use for coloring.
            Default: first 4 features.
        feature_names : list[str], optional
            Names for the features. Default: "Feature 0", "Feature 1", etc.
        point_size : int
            Size of scatter points.
        height : int
            Height of each subplot in pixels.

        Returns
        -------
        go.Figure
            Plotly figure with subplots, one per feature.

        Raises
        ------
        ValueError
            If data was not stored during fit.
        """
        if self.data is None:
            raise ValueError("No data stored. Call fit() first.")

        emb = self.embeddings[embedding_idx]

        # Default: first 4 features
        if feature_indices is None:
            n_features = min(4, self.data.shape[1])
            feature_indices = list(range(n_features))

        if feature_names is None:
            feature_names = [f"Feature {i}" for i in feature_indices]

        n_plots = len(feature_indices)
        n_cols = min(2, n_plots)
        n_rows = (n_plots + n_cols - 1) // n_cols

        fig = make_subplots(
            rows=n_rows, cols=n_cols,
            subplot_titles=feature_names
        )

        for i, feat_idx in enumerate(feature_indices):
            row = i // n_cols + 1
            col = i % n_cols + 1

            feat_values = self.data[:, feat_idx]

            fig.add_trace(
                go.Scatter(
                    x=emb[:, 0],
                    y=emb[:, 1],
                    mode='markers',
                    marker=dict(
                        size=point_size,
                        opacity=0.6,
                        color=feat_values,
                        colorscale='Viridis',
                        showscale=(i == 0)  # Show colorbar only once
                    ),
                    hovertemplate=f"{feature_names[i]}: %{{marker.color:.2f}}<extra></extra>"
                ),
                row=row, col=col
            )

        fig.update_layout(
            height=height * n_rows,
            width=400 * n_cols,
            showlegend=False,
            title_text=f"Diagnostic: Embedding {embedding_idx} colored by features"
        )
        fig.show()
        return fig


    def detect_subclusters(
        self,
        embedding_idx: int = 0,
        min_subcluster_size: int = 10
    ) -> Dict[str, Any]:
        """
        Detect subclusters within each main cluster.

        Applies HDBSCAN separately to points in each cluster to find
        potential sub-structure.

        Parameters
        ----------
        embedding_idx : int
            Which embedding to analyze.
        min_subcluster_size : int
            Minimum size for subclusters.

        Returns
        -------
        dict with keys:
            'main_labels': np.ndarray - original cluster labels
            'sub_labels': np.ndarray - subcluster labels (format: "cluster_subcluster")
            'summary': dict - {cluster_id: n_subclusters} for each cluster

        Raises
        ------
        ValueError
            If no clusterings have been computed.
        """
        if not self.clusterings:
            raise ValueError("No clusterings. Provide clusterer_class.")

        emb = self.embeddings[embedding_idx]
        main_labels = self.clusterings[embedding_idx]

        sub_labels = np.empty(len(main_labels), dtype=object)
        summary = {}

        unique_clusters = [c for c in np.unique(main_labels) if c != -1]

        for cluster_id in unique_clusters:
            mask = main_labels == cluster_id
            cluster_points = emb[mask]

            if len(cluster_points) < min_subcluster_size * 2:
                # Too small to subdivide
                sub_labels[mask] = f"{cluster_id}_0"
                summary[cluster_id] = 1
                continue

            # Apply HDBSCAN to find subclusters
            sub_clusterer = HDBSCAN(min_cluster_size=min_subcluster_size)
            sub_cluster_labels = sub_clusterer.fit_predict(cluster_points)

            n_subclusters = len(set(sub_cluster_labels)) - (1 if -1 in sub_cluster_labels else 0)
            summary[cluster_id] = max(1, n_subclusters)

            # Assign labels
            for i, (idx, sub_label) in enumerate(zip(np.where(mask)[0], sub_cluster_labels)):
                if sub_label == -1:
                    sub_labels[idx] = f"{cluster_id}_noise"
                else:
                    sub_labels[idx] = f"{cluster_id}_{sub_label}"

        # Handle noise points from main clustering
        noise_mask = main_labels == -1
        sub_labels[noise_mask] = "noise"

        return {
            'main_labels': main_labels,
            'sub_labels': sub_labels,
            'summary': summary
        }


    def draw_subclusters(
        self,
        embedding_idx: int = 0,
        min_subcluster_size: int = 10,
        hover_data: Optional[Dict[str, np.ndarray]] = None,
        point_size: int = 3,
        height: int = 400
    ) -> go.Figure:
        """
        Draw embedding with subclusters highlighted.

        Shows main clustering on the left and subclusters on the right.

        Parameters
        ----------
        embedding_idx : int
            Which embedding to analyze.
        min_subcluster_size : int
            Minimum size for subclusters.
        hover_data : dict[str, np.ndarray], optional
            Additional data to show on hover.
        point_size : int
            Size of scatter points.
        height : int
            Height of figure in pixels.

        Returns
        -------
        go.Figure
        """
        result = self.detect_subclusters(embedding_idx, min_subcluster_size)
        emb = self.embeddings[embedding_idx]

        # Convert sub_labels to numeric for coloring
        unique_subs = list(set(result['sub_labels']))
        sub_numeric = np.array([unique_subs.index(s) for s in result['sub_labels']])

        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=["Main clusters", "Subclusters"]
        )

        # Build hover
        if hover_data:
            hover_text = [
                '<br>'.join([f"{k}: {v[i]}" for k, v in hover_data.items()])
                for i in range(len(emb))
            ]
        else:
            hover_text = None

        # Main clusters
        fig.add_trace(
            go.Scatter(
                x=emb[:, 0], y=emb[:, 1],
                mode='markers',
                marker=dict(size=point_size, opacity=0.6, color=result['main_labels'], colorscale='Rainbow'),
                text=hover_text,
                hovertemplate="cluster: %{marker.color}<br>%{text}<extra></extra>" if hover_text else "cluster: %{marker.color}<extra></extra>"
            ),
            row=1, col=1
        )

        # Subclusters
        fig.add_trace(
            go.Scatter(
                x=emb[:, 0], y=emb[:, 1],
                mode='markers',
                marker=dict(size=point_size, opacity=0.6, color=sub_numeric, colorscale='Rainbow'),
                text=[f"sub: {s}" for s in result['sub_labels']],
                hovertemplate="%{text}<extra></extra>"
            ),
            row=1, col=2
        )

        # Print summary
        print(f"Subcluster summary for embedding {embedding_idx}:")
        for cluster_id, n_sub in result['summary'].items():
            print(f"  Cluster {cluster_id}: {n_sub} subcluster(s)")

        fig.update_layout(height=height, width=800, showlegend=False)
        fig.show()
        return fig


### Демонстрация на `load_digits` (1797 рукописных цифр 8×8, 10 классов)

Канонический датасет для UMAP/t-SNE. Цифры 3/5/8 и 4/9 частично перекрываются, поэтому структура 2D-эмбеддинга зависит от сида — есть что валидировать.

In [4]:
# load_digits: 64 признака (интенсивности пикселей), 10 классов
digits = load_digits()
X = StandardScaler().fit_transform(digits.data)
y_true = digits.target

validator = DimRedValidator(
    embedder_class=UMAP,
    embedder_kws={'n_neighbors': 15, 'min_dist': 0.1},
    clusterer_class=HDBSCAN,
    clusterer_kws={'min_cluster_size': 30},
    random_states=[0, 1, 2, 42, 123]
)

validator.fit(X);

Computing embedding 1/5 (rs=0)... 

C:\Users\Artem Volosatov\AppData\Roaming\Python\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


done
Computing embedding 2/5 (rs=1)... 

C:\Users\Artem Volosatov\AppData\Roaming\Python\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


done
Computing embedding 3/5 (rs=2)... 

...

done
Computing embedding 4/5 (rs=42)... 

C:\Users\Artem Volosatov\AppData\Roaming\Python\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


done
Computing embedding 5/5 (rs=123)... 

C:\Users\Artem Volosatov\AppData\Roaming\Python\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


done
Reference clustering: 10 clusters found


In [5]:
print("=" * 50)
print("1. BASIC EMBEDDINGS")
print("=" * 50)
validator.draw_embeddings([0, 1, 2], hover_data={'true_label': y_true});

1. BASIC EMBEDDINGS


In [6]:
print("\n" + "=" * 50)
print("2. CLUSTERIZED (colored by first embedding clusters)")
print("=" * 50)
validator.draw_clusterized([0, 1, 2], hover_data={'true_label': y_true});



2. CLUSTERIZED (colored by first embedding clusters)


In [7]:
print("\n" + "=" * 50)
print("3. COLORED BY COORDINATES")
print("=" * 50)
validator.draw_colored([0, 1, 2], hover_data={'true_label': y_true});



3. COLORED BY COORDINATES


In [8]:
# --- Stability analysis ---
print("\n" + "=" * 50)
print("4. STABILITY ANALYSIS (ARI)")
print("=" * 50)
stability = validator.compute_stability()
print(f"Mean ARI: {stability['mean_ari']:.4f}")
print(f"Std ARI:  {stability['std_ari']:.4f}")
print(f"Min ARI:  {stability['min_ari']:.4f}")
print(f"Max ARI:  {stability['max_ari']:.4f}")
print(f"\nARI Matrix:\n{stability['ari_matrix'].round(3)}")



4. STABILITY ANALYSIS (ARI)
Mean ARI: 0.9466
Std ARI:  0.0385
Min ARI:  0.8941
Max ARI:  0.9845

ARI Matrix:
[[1.    0.899 0.984 0.984 0.972]
 [0.899 1.    0.9   0.894 0.907]
 [0.984 0.9   1.    0.982 0.973]
 [0.984 0.894 0.982 1.    0.971]
 [0.972 0.907 0.973 0.971 1.   ]]


In [9]:
print("\n" + "=" * 50)
print("5. LEAST SIMILAR PAIR")
print("=" * 50)
validator.draw_least_similar(hover_data={'true_label': y_true});



5. LEAST SIMILAR PAIR
Least similar pair: embeddings 1 & 3, ARI = 0.8941


In [10]:
# --- Precomputed embeddings ---
print("\n" + "=" * 50)
print("6. PRECOMPUTED EMBEDDINGS (UMAP vs t-SNE)")
print("=" * 50)
umap_emb = UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X)
tsne_emb = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X)

validator_pre = DimRedValidator(
    embedder_class=UMAP,
    clusterer_class=HDBSCAN,
    clusterer_kws={'min_cluster_size': 30},
)
validator_pre.fit(X, precomputed_embeddings=[umap_emb, tsne_emb])
validator_pre.draw_clusterized([0, 1], hover_data={'true_label': y_true});



6. PRECOMPUTED EMBEDDINGS (UMAP vs t-SNE)


C:\Users\Artem Volosatov\AppData\Roaming\Python\Python313\site-packages\umap\umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



Loaded 2 precomputed embeddings
Reference clustering: 10 clusters found


In [11]:
# --- Diagnostic plots ---
print("\n" + "=" * 50)
print("7. DIAGNOSTIC PLOTS (colored by features)")
print("=" * 50)
validator.draw_diagnostic(embedding_idx=0, feature_indices=[19, 21, 28, 43]);



7. DIAGNOSTIC PLOTS (colored by features)


In [12]:
# --- Subclusters ---
print("\n" + "=" * 50)
print("8. SUBCLUSTERS")
print("=" * 50)
validator.draw_subclusters(embedding_idx=0, min_subcluster_size=15, hover_data={'true_label': y_true});


8. SUBCLUSTERS
Subcluster summary for embedding 0:
  Cluster 0: 1 subcluster(s)
  Cluster 1: 2 subcluster(s)
  Cluster 2: 2 subcluster(s)
  Cluster 3: 1 subcluster(s)
  Cluster 4: 3 subcluster(s)
  Cluster 5: 4 subcluster(s)
  Cluster 6: 4 subcluster(s)
  Cluster 7: 2 subcluster(s)
  Cluster 8: 2 subcluster(s)
  Cluster 9: 1 subcluster(s)
